In [0]:
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("load_type","")
dbutils.widgets.text("merge_keys","")

In [0]:
table_name=dbutils.widgets.get("table_name")
landing_timestamp=dbutils.widgets.get("landing_timestamp")
load_type=dbutils.widgets.get("load_type")
merge_keys=dbutils.widgets.get("merge_keys")
import pyspark.sql.functions as F


In [0]:
bronze_df=spark.read.table(f"databricks_rg.bronze.{table_name}").filter(f"landing_timestamp='{landing_timestamp}'")
updated_df=bronze_df.withColumn("insert_timestamp",F.current_timestamp()).withColumn("updated_timestamp",F.current_timestamp())


In [0]:
from delta.tables import DeltaTable

silver_table = f"databricks_rg.silver.{table_name}"

if load_type == "FULL":
    updated_df.write.mode("overwrite").saveAsTable(silver_table)

elif load_type == "APPEND":
    updated_df.write.mode("append").saveAsTable(silver_table)

elif load_type == "MERGE":
    keys = [k.strip() for k in merge_keys.split(",")]
    merge_condition = " AND ".join([f"target.{k} = source.{k}" for k in keys])

    # Update all columns on match EXCEPT insert_timestamp (preserve original record creation time)
    update_set = {col: f"source.{col}" for col in updated_df.columns if col != "insert_timestamp"}
    insert_set = {col: f"source.{col}" for col in updated_df.columns}

    if spark.catalog.tableExists(silver_table):
        delta_table = DeltaTable.forName(spark, silver_table)
        (
            delta_table.alias("target")
            .merge(updated_df.alias("source"), merge_condition)
            .whenMatchedUpdate(set=update_set)
            .whenNotMatchedInsert(values=insert_set)
            .execute()
        )
    else:
        # Target table doesn't exist yet — create it on first run
        updated_df.write.mode("overwrite").saveAsTable(silver_table)

else:
    raise ValueError(f"Unsupported load_type: '{load_type}'. Expected FULL, APPEND, or MERGE.")

In [0]:
record_return=updated_df.count()
dbutils.notebook.exit(str(record_return))